# Practica: Visualizacion de datos (Tema 3)

**Asignatura:** Analisis y visualizacion de datos (GAD-2401)

## Objetivos
1. EDA visual con Matplotlib/Seaborn.
2. PCA y MDS para proyeccion lineal.
3. Comparar con t-SNE (proyeccion no lineal).
4. Serie periodica + espectro basico (FFT).
5. Mini-tablero de resumen.

> Resuelve antes los ejercicios de `ejercicios/ejercicios-manuales.html`.


## 1. Librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import MDS, TSNE
from sklearn.datasets import load_iris

sns.set_theme(style="whitegrid")
np.random.seed(42)
print("OK")

## 2. Datos: iris

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["species"] = df["target"].map(dict(enumerate(iris.target_names)))
X = df.drop(columns=["target", "species"])
y = df["species"]
print(df.shape)
display(df.head())
display(df.describe().round(2))

## 3. EDA: pairplot y correlaciones

In [ ]:
sns.pairplot(df.drop(columns="target"), hue="species", corner=True)
plt.suptitle("Pairplot – iris", y=1.02)
plt.show()

corr = X.corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="RdBu_r", center=0, square=True)
plt.title("Correlacion entre variables")
plt.tight_layout()
plt.show()

## 4. PCA (3.1.2)

In [ ]:
Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=2)
scores = pca.fit_transform(Xs)
print("Varianza explicada:", pca.explained_variance_ratio_.round(3))
print("Acumulada:", pca.explained_variance_ratio_.sum().round(3))

pca_df = pd.DataFrame(scores, columns=["PC1", "PC2"])
pca_df["species"] = y.values

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="species", s=60)
plt.title("PCA – iris (estandarizado)")
plt.tight_layout()
plt.show()

loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=["PC1", "PC2"]
)
print("Loadings:")
display(loadings.round(3))

## 5. MDS (3.1.1)

In [ ]:
mds = MDS(n_components=2, normalized_stress="auto", random_state=42)
mds_coords = mds.fit_transform(Xs)
mds_df = pd.DataFrame(mds_coords, columns=["Dim1", "Dim2"])
mds_df["species"] = y.values
print("Stress:", round(mds.stress_, 4))

plt.figure(figsize=(7, 5))
sns.scatterplot(data=mds_df, x="Dim1", y="Dim2", hue="species", s=60)
plt.title("MDS clasico – iris")
plt.tight_layout()
plt.show()

## 6. t-SNE (3.2)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca")
tsne_coords = tsne.fit_transform(Xs)
tsne_df = pd.DataFrame(tsne_coords, columns=["tSNE1", "tSNE2"])
tsne_df["species"] = y.values

plt.figure(figsize=(7, 5))
sns.scatterplot(data=tsne_df, x="tSNE1", y="tSNE2", hue="species", s=60)
plt.title("t-SNE – iris (perplexity=30)")
plt.tight_layout()
plt.show()
print("Compara visualmente con PCA/MDS. No interpretes distancias globales en t-SNE como metricas reales.")

## 7. Serie periodica + FFT (3.3)

In [ ]:
t = np.arange(0, 365 * 2)
# Señal: ciclo anual + ciclo semanal + ruido
senal = (
    10
    + 3 * np.sin(2 * np.pi * t / 365)
    + 1.2 * np.sin(2 * np.pi * t / 7)
    + np.random.normal(0, 0.5, size=t)
)

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
axes[0].plot(t[:120], senal[:120])
axes[0].set_title("Serie sintetica (primeros 120 dias)")
axes[0].set_xlabel("dia")

# FFT
n = len(senal)
freqs = np.fft.rfftfreq(n, d=1.0)
espectro = np.abs(np.fft.rfft(senal - senal.mean()))
axes[1].plot(freqs[1:80], espectro[1:80])
axes[1].set_title("Espectro de potencia (bajas frecuencias)")
axes[1].set_xlabel("frecuencia (1/dia)")
plt.tight_layout()
plt.show()

# Periodos dominantes (excluyendo f=0)
idx = np.argsort(espectro[1:])[::-1][:5] + 1
for i in idx:
    if freqs[i] > 0:
        print(f"f={freqs[i]:.4f}  periodo≈{1/freqs[i]:.1f} dias  potencia={espectro[i]:.1f}")

## 8. Mini-tablero (resumen)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="species", ax=axes[0], legend=False)
axes[0].set_title("PCA")
sns.scatterplot(data=mds_df, x="Dim1", y="Dim2", hue="species", ax=axes[1], legend=False)
axes[1].set_title("MDS")
sns.scatterplot(data=tsne_df, x="tSNE1", y="tSNE2", hue="species", ax=axes[2])
axes[2].set_title("t-SNE")
plt.suptitle("Tablero exploratorio – proyecciones iris", y=1.02)
plt.tight_layout()
plt.show()

## 9. Ejercicios del notebook

### A
Repite PCA/MDS con el dataset `penguins` de seaborn (solo columnas numericas, dropna).

### B
Cambia `perplexity` en t-SNE (5, 30, 50) y comenta diferencias.

### C
Redacta 5 lineas de conclusiones del caso iris para un informe.


### Espacio A

In [ ]:
# Tu codigo


### Espacio B y C

B)

C)


## 10. Cierre del tema 3

| Subtema | Herramienta clave |
|---------|-------------------|
| 3.1 Lineal | PCA, MDS |
| 3.2 No lineal | t-SNE, UMAP |
| 3.3 Periodicos / tableros | FFT, dashboards |
| 3.4 Caso | Pipeline EDA + proyecciones |

**Fin de la practica del tema 3**
